In [1]:
from fast_bitrix24 import Bitrix
from datetime import datetime, timedelta
import psycopg2
import pandas as pd

Свой вебхук

In [3]:
webhook = "https://bitrix***.ru/rest/***/***/"
b = Bitrix(webhook)

Загрузка данных

In [5]:
#создаём запрос SQL
# Параметры подключения  

conn_params = {
    "host": "***.yandexcloud.net",     # Хост 
    "port": "5555",                     # Порт
    "dbname": "dwh",     # Имя базы данных
    "user": "zaboev.a",            # Имя пользователя
    "password": "****"         # Пароль
}

# Установление соединения
try:
    conn = psycopg2.connect(**conn_params)
    print("Соединение с Greenplum установлено!")
except Exception as e:
    print("Ошибка подключения:", e)

# Закрытие соединение после использования
finally:
    if 'conn' in locals() and conn:
        conn.close()
    
try:
    # Установление соединения
    conn = psycopg2.connect(**conn_params)

#агрегированные данные из DWH
    
#неделя все уровни
    sql_query_1 = """
***
"""
#неделя каждый уровень отдельно
    sql_query_2 = """
***
"""

#месяц общий результат
    sql_query_3 = """
***
"""
#месяц по уровням
    sql_query_4 = """
***
"""

# Заказы общее
    sql_query_5 = """
***
"""
# Заказы по уровням
    sql_query_6 = """
***
""" 

# Движение базы общее
    sql_query_7 = """
***
""" 
# Движение базы Премиум
    sql_query_8 = """
***
""" 
# Движение базы Бизнес
    sql_query_9 = """
***
""" 
    
    # Загрузка данных в DataFrame
    week_all = pd.read_sql_query(sql_query_1, conn) #данные продажи общие неделя
    week_lvls = pd.read_sql_query(sql_query_2, conn) #данные продажи по уровням неделя
    month_all = pd.read_sql_query(sql_query_3, conn) #данные продажи общие месяц
    month_lvls = pd.read_sql_query(sql_query_4, conn) #данные продажи по уровням месяц
    week_all_ord = pd.read_sql_query(sql_query_5, conn) #данные заказы общие неделя
    week_lvls_ord = pd.read_sql_query(sql_query_6, conn) #данные заказы по уровням месяц
    base_all = pd.read_sql_query(sql_query_7, conn) #данные движение базы общие неделя
    base_premium = pd.read_sql_query(sql_query_8, conn) #данные движение базы Премиум неделя
    base_busines = pd.read_sql_query(sql_query_9, conn) #данные движение базы Бизнес неделя

except Exception as e:
    print("Ошибка:", e)
finally:
    # Закрытие соединения
    if 'conn' in locals() and conn:
        conn.close()
        print(f"Соединение закрыто {pd.to_datetime("today")}.")

Соединение с Greenplum установлено!


C:\Users\1\AppData\Local\Temp\ipykernel_13592\2199644398.py:342: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  week_all = pd.read_sql_query(sql_query_1, conn) #данные продажи общие неделя
C:\Users\1\AppData\Local\Temp\ipykernel_13592\2199644398.py:343: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  week_lvls = pd.read_sql_query(sql_query_2, conn) #данные продажи по уровням неделя
C:\Users\1\AppData\Local\Temp\ipykernel_13592\2199644398.py:344: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  month_all = pd.read_sql_query(sql_qu

Соединение закрыто 2026-02-16 18:13:56.705125.


In [6]:
#меняем тип данных в недельной общей таблице
week_all['rep_dt'] = pd.to_datetime(week_all['rep_dt'])

Функции для переменных строк

In [32]:
# функция положительного или отрицательного
def balance(value):
    if value > 0:
        value_str = f"+{value}"
    elif value < 0:
        value_str = str(value)
    elif value == 0.0:
        value_str = ("0")
    return value_str

In [9]:
# функция перевыполнение или недовыполнение для АКБ месяц
def plan_ex(value, plan):
    if value > plan:
        diff = value - plan
        value_str = f"перевыполнение на {diff} клиентов"
    elif value < plan:
        diff = plan - value
        value_str = f"недовыполнение на {diff} клиентов"
    elif value == plan:
        value_str = "выполнит план"
    return value_str

In [10]:
# функция снижение или увеличение тип данных float для общих заказов
def up_down(value):
    value = float(value)
    if value > 0 and value >= 0.5:
        value_str = "увеличение до"
    elif value < 0 and value <= -0.5:
        value_str = "снижение до"
    else:
        value_str = "стабильность"
    return value_str

In [13]:
# функция снижение или увеличение int для АКБ заказов
def up_down_int(value):
    if value > 0 and value >= 3:
        value_str = "с ростом"
    elif value < 0 and value <= -3:
        value_str = "со снижением"
    else:
        value_str = "стабильно"
    return value_str

In [14]:
# функция снижение или увеличение простое для ср.суммы заказа
def up_down_AVG(value):
    value = float(value)
    if value > 0:
        value_str = "с ростом"
    elif value < 0:
        value_str = "со снижением"
    else:
        value_str = "стабильна"
    return value_str

In [15]:
# функция снижение или увеличение простое для движения базы
def get_trend(delta):
    delta = float(delta)
    if delta > 0:
        return "увеличение"
    elif delta < 0:
        return "уменьшение"
    else:
        return "без изменений"

Все переменные, используемые в отчёте

In [38]:
# Недельные продажи общие и динамика продаж
w_sales_all = int(week_all.sort_values(by='rep_dt', ascending=False).head(1)['sales_amt'].iloc[0])
w_sales_dynamic = balance(week_all.sort_values(by='rep_dt', ascending=False).head(1)['sales_dyn'].iloc[0])

#недельные продажи по уровням и динамика продаж
w_sales_premium = int(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['sales_amt'].iloc[0])
w_sales_premium_dynamic = balance(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['sales_dyn'].iloc[0])
w_sales_busines = int(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['sales_amt'].iloc[0])
w_sales_busines_dynamic = balance(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['sales_dyn'].iloc[0])

#месячные продажи -общий прогноз, выполнение полана
s_forecast_all = int(month_all['sales_forecast_amt'].iloc[0])
s_exec_all = int(month_all['sales_amt_exec'].iloc[0])
#месячные продажи - выполнение полана по уровням
s_exec_b = int(month_lvls.query("loyalty_lvl == 'Бизнес'")['sales_amt_exec'].iloc[0])
s_exec_p = int(month_lvls.query("loyalty_lvl == 'Премиум'")['sales_amt_exec'].iloc[0])

#недельные АКБ по уровням и динамика АКБ
w_ACB_premium = int(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['unit_cvm_cnt'].iloc[0])
w_ACB_premium_dynamic = balance(int(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['unit_cvm_dyn'].iloc[0]))
w_ACB_busines = int(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['unit_cvm_cnt'].iloc[0])
w_ACB_busines_dynamic = balance(int(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['unit_cvm_dyn'].iloc[0]))

#месячная АКБ - прогноз общий, прогноз по уровням, исполнение плана общее
AKB_forecast_all = int(month_all['unit_cvm_cnt_forecast'].iloc[0])
AKB_forecast_p = int(month_lvls.query("loyalty_lvl == 'Премиум'")['unit_cvm_cnt_forecast'].iloc[0])
AKB_forecast_b = int(month_lvls.query("loyalty_lvl == 'Бизнес'")['unit_cvm_cnt_forecast'].iloc[0])
AKB_exec_all = int(month_all['unit_cvm_cnt_exec'].iloc[0])
#месячная АКБ по уроням - перевыполнение/недовыполнение планов на сколько клиентов
AKB_diff_plan_p = plan_ex(int(month_lvls.query("loyalty_lvl == 'Премиум'")['unit_cvm_cnt_forecast'].iloc[0]), int(month_lvls.query("loyalty_lvl == 'Премиум'")['active_unit_cvm_plan_cnt'].iloc[0]))
AKB_diff_plan_b = plan_ex(int(month_lvls.query("loyalty_lvl == 'Бизнес'")['unit_cvm_cnt_forecast'].iloc[0]), int(month_lvls.query("loyalty_lvl == 'Бизнес'")['active_unit_cvm_plan_cnt'].iloc[0]))

#недельная АРПУ по уровням и динамика 
w_ARPU_premium = int(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['arpu'].iloc[0])
w_ARPU_premium_dynamic = balance(int(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['arpu_dyn'].iloc[0]))
w_ARPU_busines = int(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['arpu'].iloc[0])
w_ARPU_busines_dynamic = balance(int(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['arpu_dyn'].iloc[0]))

#месячная АРПУ по уровням - прогноз, выполнение плана
m_ARPU_premium_forcast = int(month_lvls.query("loyalty_lvl == 'Премиум'")['arpu_forecast'].iloc[0])
m_ARPU_busines_forcast = int(month_lvls.query("loyalty_lvl == 'Бизнес'")['arpu_forecast'].iloc[0])
m_ARPU_exec_p = int(round(month_lvls.query('loyalty_lvl == "Премиум"')['arpu_forecast'] / month_lvls.query('loyalty_lvl == "Премиум"')['arpu_plan_amt']*100).iloc[0])
m_ARPU_exec_b = int(round(month_lvls.query('loyalty_lvl == "Бизнес"')['arpu_forecast'] / month_lvls.query('loyalty_lvl == "Бизнес"')['arpu_plan_amt']*100).iloc[0])

#недельная ВП общая, по уровням, динамика по уровням
w_profit_all = week_all.sort_values(by='rep_dt', ascending=False).head(1)['profit_amt'].iloc[0]
w_profit_premium = week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['profit_amt'].iloc[0]
w_profit_premium_dynamic = balance(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['profit_dyn'].iloc[0])
w_profit_busines = week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['profit_amt'].iloc[0]
w_profit_busines_dynamic = balance(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['profit_dyn'].iloc[0])

#месячный ВП прогноз общий, по уровням, выполнение плана общее
profit_forecast_all = month_all['profit_forecast_amt'].iloc[0]
profit_exec_all = int(month_all['profit_amt_exec'].iloc[0])
profit_forecast_b = month_lvls.query("loyalty_lvl == 'Бизнес'")['profit_forecast_amt'].iloc[0]
profit_forecast_p = month_lvls.query("loyalty_lvl == 'Премиум'")['profit_forecast_amt'].iloc[0]

#Маржа недельная общая, по уровням, динамика
marge = week_all.sort_values(by='rep_dt', ascending=False).head(1)['marge'].iloc[0]
marge_p = week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['marge'].iloc[0]
marge_dyn_p = balance(week_lvls.query("loyalty_lvl == 'Премиум'").sort_values(by='rep_dt', ascending=False).head(1)['marge_dyn'].iloc[0])
marge_b = week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['marge'].iloc[0]
marge_dyn_b = balance(week_lvls.query("loyalty_lvl == 'Бизнес'").sort_values(by='rep_dt', ascending=False).head(1)['marge_dyn'].iloc[0])

# заказы (всегда недельные) сумма, динамика
w_orders_all = int(week_all_ord.sort_values(by='rep_week', ascending=False).head(1)['сумма'].iloc[0])
w_orders_all_dyn = balance(week_all_ord.sort_values(by='rep_week', ascending=False).head(1)['изменение'].iloc[0])

# количество заказов по уровням, динамика
w_orders_cnt_p = int(week_lvls_ord.query("loyalty_lvl == 'Премиум'").sort_values(by='дата', ascending=False).head(1)['количество'].iloc[0])
w_orders_cnt_p_dyn = balance(week_lvls_ord.query("loyalty_lvl == 'Премиум'").sort_values(by='дата', ascending=False).head(1)['количество_rate'].iloc[0])
w_orders_cnt_b = int(week_lvls_ord.query("loyalty_lvl == 'Бизнес'").sort_values(by='дата', ascending=False).head(1)['количество'].iloc[0])
w_orders_cnt_b_dyn = balance(week_lvls_ord.query("loyalty_lvl == 'Бизнес'").sort_values(by='дата', ascending=False).head(1)['количество_rate'].iloc[0])

#заказы на 1 АКБ общие, динамика
w_orders_1AKB_all =  week_all_ord.sort_values(by='rep_week', ascending=False).head(1)['количество_на_1_АКБ'].iloc[0]
w_orders_1AKB_all_dyn = balance(week_all_ord.sort_values(by='rep_week', ascending=False).head(1)['изменение_1АКБ'].iloc[0])

# АКБ заказов по уровням, динамика, общая динамика
w_orders_AKB_p = int(week_lvls_ord.query("loyalty_lvl == 'Премиум'").sort_values(by='дата', ascending=False).head(1)['АКБ'].iloc[0])
w_orders_AKB_p_dyn = balance(int(week_lvls_ord.query("loyalty_lvl == 'Премиум'").sort_values(by='дата', ascending=False).head(1)['АКБ_dyn'].iloc[0]))
w_orders_AKB_b = int(week_lvls_ord.query("loyalty_lvl == 'Бизнес'").sort_values(by='дата', ascending=False).head(1)['АКБ'].iloc[0])
w_orders_AKB_b_dyn = balance(int(week_lvls_ord.query("loyalty_lvl == 'Бизнес'").sort_values(by='дата', ascending=False).head(1)['АКБ_dyn'].iloc[0]))
#
w_orders_AKB_all_dyn = int(week_all_ord.sort_values(by='rep_week', ascending=False).head(1)['изменение_АКБ'].iloc[0])

#средняя сумма заказа общая динамика, по уровням + динамика
w_orders_AVG_SUM_all_dyn = week_all_ord.sort_values(by='rep_week', ascending=False).head(1)['изменение_ср_суммы'].iloc[0]
#
w_orders_AVG_SUM_p = week_lvls_ord.query("loyalty_lvl == 'Премиум'").sort_values(by='дата', ascending=False).head(1)['средняя_сумма_заказа'].iloc[0]
w_orders_AVG_SUM_p_dyn = balance(week_lvls_ord.query("loyalty_lvl == 'Премиум'").sort_values(by='дата', ascending=False).head(1)['средняя_сумма_заказа_dyn'].iloc[0])
w_orders_AVG_SUM_b = week_lvls_ord.query("loyalty_lvl == 'Бизнес'").sort_values(by='дата', ascending=False).head(1)['средняя_сумма_заказа'].iloc[0]
w_orders_AVG_SUM_b_dyn = balance(week_lvls_ord.query("loyalty_lvl == 'Бизнес'").sort_values(by='дата', ascending=False).head(1)['средняя_сумма_заказа_dyn'].iloc[0])

#Движение базы общая дельта, дельта по уровням, изменение дельты от неделе к неделе
base_delta_all = balance(base_all['Дельта'].iloc[0])
base_delta_p = balance(base_premium.sort_values(by='Дата', ascending=False).head(1)['Дельта'].iloc[0])
delta_change_p = base_premium.sort_values(by='Дата', ascending=False).head(1)['Дельта'].iloc[0] - base_premium.sort_values(by='Дата', ascending=True).head(1)['Дельта'].iloc[0]
base_delta_b = balance(base_busines.sort_values(by='Дата', ascending=False).head(1)['Дельта'].iloc[0])
delta_change_b = base_busines.sort_values(by='Дата', ascending=False).head(1)['Дельта'].iloc[0] - base_busines.sort_values(by='Дата', ascending=True).head(1)['Дельта'].iloc[0]

Работа с датами

In [20]:
# календарь с русскими названиями для указания месяца в тексте
MONTHS_RU = {
    1: 'январь',
    2: 'февраль',
    3: 'март',
    4: 'апрель',
    5: 'май',
    6: 'июнь',
    7: 'июль',
    8: 'август',
    9: 'сентябрь',
    10: 'октябрь',
    11: 'ноябрь',
    12: 'декабрь'
}

In [21]:
# опеределение сегодняшнего дня, месяца, превращение дня в удобно читаемую строку
today = datetime.today().date()
month_name = MONTHS_RU[today.month]
today = today.strftime('%d.%m.%Y')

Сам доклад

In [44]:

report = (
    f"[B]Отчёт на {today}:[/B]\n \n"
    f"1. Показатель недельной выручки составил [B]{w_sales_all} млн[/B] ([B]{w_sales_dynamic} млн[/B]). Премиум принёс [B]{w_sales_premium} млн[/B] ([B]{w_sales_premium_dynamic} млн[/B]), Бизнес [B]{w_sales_busines} млн[/B] ([B]{w_sales_busines_dynamic} млн)[/B].\n" 
    f"2. Прогноз по выручке – [B]{s_forecast_all} млн[/B], то есть [B]{s_exec_all}%[/B] от плана. Премиум идёт на [B]{s_exec_p}%[/B], Бизнес на [B]{s_exec_b}%[/B].\n"
    f"3. В АКБ Премиум за неделю [B]{w_ACB_premium_dynamic} клиентов[/B], показатель [B]{w_ACB_premium}[/B]. Бизнес [B]{w_ACB_busines} клиентов[/B] ([B]{w_ACB_busines_dynamic}[/B] за неделю).\n" 
    f"4. Прогноз по АКБ [B]{AKB_forecast_all} клиента[/B] за декабрь, из них [B]{AKB_forecast_p}[/B] Премиум и [B]{AKB_forecast_b}[/B] Бизнес. Прогнозируется выполнение плана на [B]{AKB_exec_all}%[/B]. Премиум прогнозно - [B]{AKB_diff_plan_p}[/B], Бизнес - [B]{AKB_diff_plan_b}[/B].\n" 
    f"5. АРПУ за неделю: Премиум [B]{w_ARPU_premium} тыс.[/B] ([B]{w_ARPU_premium_dynamic} тыс.[/B]), Бизнес [B]{w_ARPU_busines} тыс.[/B] ([B]{w_ARPU_busines_dynamic} тыс.[/B]).\n"
    f"6. Прогнозируемая АРПУ Премиум за {month_name} составит [B]{m_ARPU_premium_forcast} тыс.[/B] ([B]{m_ARPU_exec_p}%[/B] от плана), Бизнес [B]{m_ARPU_busines_forcast} тыс.[/B] ([B]{m_ARPU_exec_b}%[/B]).\n"
    f"7. ВП составляет [B]{w_profit_all} млн[/B] за неделю. По уровням: Премиум [B]{w_profit_premium} млн[/B] ([B]{w_profit_premium_dynamic} млн[/B]), Бизнес [B]{w_profit_busines} млн [/B] ([B]{w_profit_busines_dynamic} млн[/B]).\n"
    f"8. Прогнозно по ВП идём на [B]{profit_forecast_all} млн[/B] ([B]{profit_forecast_p} млн[/B] – Премиум, [B]{profit_forecast_b} млн[/B] - Бизнес). Ожидается выполнение плана на [B]{profit_exec_all}%[/B].\n"
    f"9. Маржинальная прибыль составила [B]{marge_p}%[/B] в Премиум ([B]{marge_dyn_p}%[/B]) и [B]{marge_b}%[/B] в Бизнес ([B]{marge_dyn_b}%[/B]). Общий показатель за неделю [B]{marge}%[/B].\n"
    f"10. В заказах отмечается {up_down(w_orders_all_dyn)} [B]{w_orders_all} млн[/B] ([B]{w_orders_all_dyn} млн[/B] к прошлой неделе).\n"
    f"11. Количество заказов по уровням: Премиум [B]{w_orders_cnt_p}[/B] ([B]{w_orders_cnt_p_dyn} %[/B]), Бизнес [B]{w_orders_cnt_b}[/B] ([B]{w_orders_cnt_b_dyn}%[/B]). \n"
    f"12. АКБ заказов - {up_down_int(w_orders_AKB_all_dyn)}. У Премиум [B]{w_orders_AKB_p} клиентов[/B]  ([B]{w_orders_AKB_p_dyn}[/B]), у Бизнес [B]{w_orders_AKB_b} клиентов[/B] ([B]{w_orders_AKB_b_dyn}[/B]). \n"
    f"13. Средняя сумма заказов - {up_down_AVG(w_orders_AVG_SUM_all_dyn)}. В Премиум составляет [B]{w_orders_AVG_SUM_p} тыс.[/B] ([B]{w_orders_AVG_SUM_p_dyn} тыс.[/B]), в Бизнес [B]{w_orders_AVG_SUM_b} тыс.[/B] ([B]{w_orders_AVG_SUM_b_dyn} тыс.[/B]). \n"
    f"14. Общий показатель среднего количества заказов на 1 АКБ равен [B]{w_orders_1AKB_all}[/B] ([B]{w_orders_1AKB_all_dyn}[/B] к прошлой неделе). \n"
    f"15. В движении базы общая дельта оттока-реактивации составляет [B]{base_delta_all}%[/B], в Премиуме дельта [B]{base_delta_p}%[/B] ({get_trend(base_delta_p)} с прошлой недели). В Бизнесе [B]{base_delta_b}%[/B] ({get_trend(base_delta_b)} с прошлой недели).\n\n"

    "[URL=https://datalens.ru/***] Дашборд основных показателей[/URL]"
)

In [46]:
# Отправка в чат Битры
b.call('im.message.add', {
    'DIALOG_ID': 'chat00000',
    'MESSAGE': report
})

<Task pending name='Task-8' coro=<log.<locals>.wrapper() running at C:\Users\1\anaconda3\Lib\site-packages\fast_bitrix24\logger.py:9>>

  0%|          | 0/1 [00:00<?, ?it/s]